# MSDM 5002 HW_2

Author：    LAN Tianwei 藍天蔚<br>
ID：        21230969<br>
Email：     tlanaa@connect.ust.hk<br>

# Q1: tic-tac-toe

## Part a: check winner

In [46]:
def check_winner(board):
    if type(board) != list or len(board) != 9:
        return "board must be a list with 9 elements"
    for x in board:
        if x not in (-1,0,1):
           return "board's elements must be -1, 0 or 1"

    winning = [[0,1,2],[3,4,5],[6,7,8],
               [0,3,6],[1,4,7],[2,5,8],
               [0,4,8],[2,4,6]]
    for w in winning:
        i,j,k = w
        if board[i]+board[j]+board[k]==3:
            return 1
        elif board[i]+board[j]+board[k]==-3:
            return -1
    return 0

In [47]:
### test for check_winner
# ---------- Normal ----------
if __name__ == '__main__':
    print(check_winner([ 1, 1, 1,
                        0,-1, 0,
                        -1, 0,-1])) # A wins 
    print(check_winner([-1, 1, 0,
                        -1, 0, 1,
                        -1, 0, 1])) # B wins
    print(check_winner([ 1, 0, 0,
                        0, 1,-1,
                        -1, 0,-1])) # Draw 
    print(check_winner([0]*9))      # Empty

    # ---------- invalid ----------
    print(check_winner("not a list"))  # Wrong type
    print(check_winner([1, 0, -1]))    # Wrong length
    print(check_winner([ 1, 0, 0,
                        0, 2,-1,
                        -1, 0,-1]))    # Invalid value


1
-1
0
0
board must be a list with 9 elements
board must be a list with 9 elements
board's elements must be -1, 0 or 1


## Part b: random simulation

At first, I tried to directly generate a board with five 1s and four -1s to represent a complete game state, but I quickly realized this was flawed: a fully filled board cannot capture the fact that the game ends immediately when one player wins. This approach produces many illegal scenarios—such as both players winning, or a win being "delayed" beyond the actual winning move. Therefore, I switched to a step-by-step simulation instead.

In [49]:
import random

def simulate_ttt():
    board = [0]*9
    player = 1
    for move in range(9):
        empty = [i for i,x in enumerate(board) if x==0] # get empty index
        choice = random.choice(empty)
        board[choice] = player
        winner = check_winner(board)
        if winner != 0:
            return winner
        player = -player # switch player
    return 0 # draw

if __name__ == '__main__':
    N = 1000
    cnt_a = 0
    cnt_b = 0
    cnt_draw = 0
    for i in range(N):
        win = simulate_ttt()
        if win == 1:
            cnt_a += 1
        elif win == -1:
            cnt_b += 1
        elif win == 0: 
            cnt_draw += 1
    print(f"A wins {cnt_a} times")
    print(f"B wins {cnt_b} times")
    print(f"draw for {cnt_draw} times")


A wins 569 times
B wins 293 times
draw for 138 times


We can tell from the repeat experiments that A wins much more than B. The results are consistent with the win rate calculated by exact recursive enumeration.

In [ ]:
# Recursive enumeration, Generated by AI

import itertools
import numpy as np

# We'll enumerate all possible games with random play (uniform random among available moves)
# But since the policy is random, each complete game path has a probability = product of 1/(available moves at each step)

# We can do a recursive search to compute exact probabilities.

def get_winner(board):
    # board is a tuple of 9 elements: 0=empty, 1=A, 2=B
    lines = [
        (0,1,2), (3,4,5), (6,7,8), # rows
        (0,3,6), (1,4,7), (2,5,8), # cols
        (0,4,8), (2,4,6)           # diagonals
    ]
    for a,b,c in lines:
        if board[a] != 0 and board[a] == board[b] == board[c]:
            return board[a]
    if 0 not in board:
        return 0  # draw
    return None  # game not over

from functools import lru_cache

@lru_cache(maxsize=None)
def solve(board, player):
    # board: tuple of 9 ints (0,1,2)
    # player: 1 or 2 (whose turn it is)
    # Returns (pA_win, pB_win, pDraw) from this state under random play
    winner = get_winner(board)
    if winner is not None:
        if winner == 1:
            return (1.0, 0.0, 0.0)
        elif winner == 2:
            return (0.0, 1.0, 0.0)
        else: # draw
            return (0.0, 0.0, 1.0)
    
    empty_positions = [i for i, x in enumerate(board) if x == 0]
    n = len(empty_positions)
    if n == 0:
        return (0.0, 0.0, 1.0)
    
    total_pA, total_pB, total_draw = 0.0, 0.0, 0.0
    for pos in empty_positions:
        new_board = list(board)
        new_board[pos] = player
        pA, pB, pD = solve(tuple(new_board), 3 - player)  # switch player
        total_pA += pA
        total_pB += pB
        total_draw += pD
    
    # Since each move is chosen uniformly at random
    return (total_pA / n, total_pB / n, total_draw / n)

# Start from empty board, player 1 (A) goes first
initial_board = (0,)*9
pA, pB, pDraw = solve(initial_board, 1)
print(f"A win: {pA:.3g}")
print(f"B win: {pB:.3g}")
print(f"draw:  {pDraw:.3g}")

A win: 0.585
B win: 0.288
draw:  0.127


## Part c: clever simulation

In [63]:
def one_move(chess,board):
    winning = [[0,1,2],[3,4,5],[6,7,8],
               [0,3,6],[1,4,7],[2,5,8],
               [0,4,8],[2,4,6]]
    for w in winning:
        i,j,k = w
        # win first, defend later
        if board[i]+board[j]+board[k] == 2*chess:       # 2 for A and -2 for B, next step is to win
            zero_idx = w[[board[i], board[j], board[k]].index(0)]
            board[zero_idx] = chess
            return board
        elif board[i]+board[j]+board[k] == -2*chess:    # -2 for A and 2 for B, next step is to lose
            zero_idx = w[[board[i], board[j], board[k]].index(0)]
            board[zero_idx] = chess
            return board
    # after walking through, no chance to lose or to win, then move randomly on blank grid
    empty = [r for r in range(9) if board[r] == 0]
    r = random.choice(empty)
    board[r] = chess
    return board
    
def simulate_clever_ttt():
    board = [0]*9
    player = 1
    for m in range(9):
        board = one_move(player,board)
        if check_winner(board) != 0:
            return player
        player = -player
    return 0

if __name__ == '__main__':
    N = 1000
    cnt_a = 0
    cnt_b = 0
    cnt_draw = 0
    for i in range(N):
        win = simulate_clever_ttt()
        if win == 1:
            cnt_a += 1
        elif win == -1:
            cnt_b += 1
        elif win == 0: 
            cnt_draw += 1
    print(f"A wins {cnt_a} times")
    print(f"B wins {cnt_b} times")
    print(f"draw for {cnt_draw} times")

A wins 287 times
B wins 156 times
draw for 557 times


A still wins more than B, but the number of draws increases significantly. This is understandable because A, by taking the initiative, is able to seize more winning opportunities, but active defense makes it easier for both parties to reach a draw.

# Q2: Blind box

In [30]:
import random

def simulate_blind_box(p):
    length = len(p)
    cnt = [0] * length
    remains = length
    total = 0
    while remains:
        idx = random.choices(range(length), weights=p, k=1)[0]
        if cnt[idx] == 0:     # 1st time drawing out this kind
            remains -= 1
        cnt[idx] += 1
        total += 1
    return total

if __name__ == '__main__':
    p = [0.05,0.1,0.15,0.2,0.2,0.3]
    n = 1000
    avg = sum(simulate_blind_box(p) for _ in range(n)) / n
    print(f"{avg:.3g}")

24.9


Actually, this is a typical "Generalized Coupon Collector Problem". Set the random variable $T$ as the minimum number of boxes required to draw all blind boxes, the we have:
$$E(T)=\sum_{\varnothing \ne S \subseteq \{1,2,...,n\}} (-1)^{|S|+1} \cdot \frac{1}{\sum_{i \in S}p_i}$$
where S is all non-empty subsets of the state set consisting of whether or not to draw each blind box. Therefore, there are a total of $2^6-1=63$ subsets.

The inverse of the probability of drawing each S is the expected number of boxes for that S. Using the inclusion-exclusion principle, we can calculate the expected number of boxes drawn from all blind boxes, which is consistent with the simulation result.

In [66]:
# Inclusion–Exclusion Principle, Generated by AI

import itertools
import math

# Probabilities for each of the six styles
probabilities = [0.05, 0.1, 0.15, 0.2, 0.2, 0.3]

# Function to calculate the expected number of boxes to collect all styles
def expected_boxes_to_collect_all(probabilities):
    n = len(probabilities)
    # Using inclusion-exclusion principle to calculate the expected value
    expected_value = 0
    # Iterate over all non-empty subsets of styles
    for k in range(1, n + 1):
        for subset in itertools.combinations(range(n), k):
            # Probability of not getting any of the styles in the subset in one draw
            prob_not_in_subset = 1 - sum(probabilities[i] for i in subset)
            # Add or subtract the reciprocal of this probability based on the inclusion-exclusion principle
            expected_value += (-1)**(k + 1) / (1 - prob_not_in_subset)
    return expected_value

expected_boxes = expected_boxes_to_collect_all(probabilities)
expected_boxes

25.139350415666186

# Q3: Looking for each other at a supermarket

## Part a: What is in common mathematically about the coordinates of the person where he is allowed to walk in different directions?

When one walking horizontally, his one-step movement will be [1,0] or [-1,0].<br>
When one walking vertically, his movement will be [0,1] or [0,-1].<br>
When one walking diagonally, his movement will be [1,1] or [-1,-1].<br>

## Part b: one step function

In [144]:
import random

def coord_check(coord):
    x,y = coord
    if (x%3!=0 and y%3!=0) and abs(x-y)/3 not in range(6):
        return False
    elif (x<0 or y<0 or x>15 or y>15):
        return False
    return True

def step(x,y,memory):
    walk = [[1,0],[0,1],[1,1],[-1,0],[0,-1],[-1,-1]]
    if (coord_check(memory) and coord_check([x,y])) == False : # invalid coordinate check
        print("invalid coordinate")
        return False
    elif ([x-memory[0],y-memory[1]] not in walk): # invalid movement check
        print("invalid move")
        return False
    
    next = [[x + dx, y + dy] for dx, dy in walk]

    for i in range(len(walk)):
        if coord_check(next[i]) == False or next[i] == memory: # delete invalid next step and memory
            next[i] = None
        # One can only go straight forward in the corridor, so check whether the mean of memory and next is [x,y]
        elif abs(x-y)%3 == 0 and x%3 != 0: # diagonal corridor
            if 2*x != next[i][0] + memory[0] or 2*y != next[i][1] + memory[1]:
                next[i] = None 
        elif abs(x-y)%3 != 0 and (x%3 == 0 or y%3 == 0): # vertical/horizontal corridor
            if 2*x != next[i][0] + memory[0] or 2*y != next[i][1] + memory[1]:
                next[i] = None

    next = [p for p in next if p is not None] # drop the Nones
    # print(next)
    idx = random.randrange(len(next))
    return next[idx][0],next[idx][1],[x,y]
    

In [145]:
# test for step(x,y,memory)                              next:
if __name__ == '__main__':
    print(step(3,6,[2,5]))      # 6 way junction         [[4, 6], [3, 7], [4, 7], [2, 6], [3, 5]]
    print(step(0,6,[0,5]))      # side 4 way junction    [[1, 6], [0, 7], [1, 7]]
    print(step(7,3,[6,3]))      # horizontal corridor    [[8, 3]]
    print(step(0,4,[0,3]))      # vertical corridor      [[0, 5]]
    print(step(1,4,[2,5]))      # diagonal corridor      [[0, 3]]
    print(step(14,0,[15,0]))    # A start                [[13, 0]]
    print(step(1,15,[0,15]))    # B start                [[2, 15]]


(4, 7, [3, 6])
(1, 7, [0, 6])
(8, 3, [7, 3])
(0, 5, [0, 4])
(0, 3, [1, 4])
(13, 0, [14, 0])
(2, 15, [1, 15])


## Part c: simulate supermarket meeting

In [154]:
def in_line(xa,xb,ya,yb):
    if (xa==xb and xa%3==0): # on the same ho corridor
        return "h"
    elif (ya==yb and ya%3==0): # on the same ve corridor
        return "v"
    elif (xa-ya)/3 == (xb-yb)/3 and (xa-ya)%3 == 0: # on the same di corridor
        return "d"
    return "n"

def simulate_supermarket():
    cnt_steps = 2
    xa,ya = random.choice([[15,1],[14,0]])
    memory_a = [15,0]
    xb,yb = random.choice([[1,15],[0,14]])
    memory_b = [0,15]
    cor_type = "n"
    
    while cor_type == "n":
        if cnt_steps % 2 == 0:
            xa,ya,memory_a = step(xa,ya,memory_a)
        elif cnt_steps % 2 == 1:
            xb,yb,memory_b = step(xb,yb,memory_b)
        cnt_steps += 1
        cor_type = in_line(xa,xb,ya,yb)

    if cor_type == "h" or cor_type == "d":
        cnt_steps += abs(ya-yb)
    elif cor_type == "v":
        cnt_steps += abs(xa-xb)
    
    return cnt_steps

if __name__ == '__main__':
    sum_steps = 0
    for i in range(1000):
        sum_steps += simulate_supermarket()
    print(sum_steps/1000)

58.393


Since the conditions of the problem are relatively complex, it is impossible to calculate the exact solution of the number of encounter steps like the previous two problems.

# Q4: trick in Russian roulette

In the case of a single bullet, I use *numpy* to generate the bullet position at once, and use the *isin()* function to determine whether it has been fired, without using a *for* loop.

In [ ]:
# single bullet simulation

import numpy as np
import time

num_test=10000
num_pos=6             # maximum number of bullets in the gun
pos_take=[1,3,4]      # the orders taking shoots for the player

start_time=time.time()

# use numpy to set bullets' position at one time
bullet = np.random.randint(0, num_pos, size=num_test)
# use isin() to check how many bullet are in [1,3,4] at one time, then calculate the mean of 0&1s into the probability
p_lose = np.isin(bullet, pos_take).mean()

print("The lose probability is:", p_lose)
print("Time:",time.time()-start_time)

The lose probability is: 0.5029
Time: 0.001003265380859375


## Bonus 1: Rewrite for the multiple bullets cases as well.

Below I wrote a Russian roulette simulation function that takes the firing position, number of loops, and number of bullets as input and returns the probability of losing.

In [5]:
# Russian roulette simulation (single & multiple bullets)
import numpy as np
import time

def rus_rlt(bullet_cnt: int, pos_take, num_test: int = 10000, num_pos: int = 10) -> float:
    if not (1 <= bullet_cnt < num_pos):
        raise ValueError("bullet_cnt only accept integer between 1 and num_pos-1 as input") 

    start_time=time.time()
    pos_take = [x - 1 for x in pos_take]

    # bullets should be Non-repeating
    # generate a 10000×6 [0,6) random permutation matrix and take its former 2 columns as bullet positions
    bullets =  np.random.rand(num_test, num_pos).argsort(axis=1)[:, :bullet_cnt]
    first_bullet = bullets.min(axis=1)

    # If the first bullet fires before your turn, you win (game ends early).
    # E.g., pos_take=[2,3,4], bullets=[1,3] → you win, not lose.
    valid_mask = first_bullet >= pos_take[0]
    valid_first_bullet = first_bullet[valid_mask]
    p_lose = np.isin(valid_first_bullet, pos_take).mean()

    return p_lose,time.time()-start_time

if __name__ == "__main__":
    p_lose, t = rus_rlt(3, [1, 2, 3, 4, 5], 10000, 10)
    print("First to shoot 5 shots")
    print("The lose probability is:",p_lose) 
    print("Time:",t)
    p_lose, t = rus_rlt(3, [2, 3, 4, 5, 6], 10000, 10)
    print("5 shots after the first")
    print("The lose probability is:",p_lose) 
    print("Time:",t)
                        
                        



First to shoot 5 shots
The lose probability is: 0.9171
Time: 0.00543522834777832
5 shots after the first
The lose probability is: 0.9530649978665908
Time: 0.003996133804321289


## Bonus 2: Explain why order matters in the multiple bullet case.

With only one bullet, Russian roulette exhibits a rising conditional probability of death after each survived shot, yet the two players still share every upcoming shot equally.

With multiple bullets (e.g., 2–5 in a six-chamber cylinder), the first shot may kill; if it does not, the positions of the remaining bullets no longer affect the overall death probability. Letting $Y$ denote the location of the first bullet, we have
$$P(Y = y) = C(n−y, k−1) / C(n,k)$$
fixed y must be chosen, leaving k−1 bullets to be placed among the larger positions y+1,…,n

A quick check shows that as k approaches n, this distribution shifts sharply toward the earlier chambers.

Thus, in the multi-bullet game the early shooter bears a higher unconditional risk, but if he survives, the late shooter faces a higher conditional probability of death. Hence, the order of firing is critical in multi-bullet Russian roulette.